# NetFury - Analisis de Benchmarks ISP Ecuador

**Proyecto:** Extraccion automatizada de planes de internet de ISPs ecuatorianos.

**Estrategias comparadas:**
| Estrategia | Descripcion | Costo |
|---|---|---|
| `benchmark-full` | HTML rendering directo (tabs interactivos) | FREE |
| `benchmark-recursive` | Crawl recursivo desde homepage (solo HTML) | FREE |
| `benchmark-recursive-images` | Crawl recursivo + analisis de imagenes (OCR/LLM) | FREE (OCR) / API (LLM) |

**ISPs monitoreados:** Netlife, Ecuanet, Claro, CNT, Xtrim, Puntonet, Alfanet, Fibramax

---

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
from pathlib import Path

plt.rcParams.update({
    "figure.facecolor": "#0f1117",
    "axes.facecolor": "#1a1d27",
    "axes.edgecolor": "#2e3348",
    "axes.labelcolor": "#e2e4ed",
    "text.color": "#e2e4ed",
    "xtick.color": "#8b8fa7",
    "ytick.color": "#8b8fa7",
    "grid.color": "#2e3348",
    "grid.alpha": 0.5,
    "figure.dpi": 120,
    "font.size": 10,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
})

COLORS = {
    "accent": "#6c5ce7",
    "accent_light": "#a29bfe",
    "green": "#00cec9",
    "green_light": "#55efc4",
    "red": "#ff6b6b",
    "yellow": "#feca57",
    "blue": "#54a0ff",
    "orange": "#ffa502",
    "pink": "#fd79a8",
    "gray": "#636e72",
}

ISP_COLORS = {
    "netlife": "#6c5ce7",
    "ecuanet": "#00cec9",
    "claro": "#ff6b6b",
    "cnt": "#feca57",
    "xtrim": "#54a0ff",
    "puntonet": "#ffa502",
    "alfanet": "#fd79a8",
    "fibramax": "#55efc4",
}

DATA_DIR = Path("../data/processed")
print("NetFury Analysis - Setup OK")

## 1. Carga de Datos

In [ ]:
# --- Summaries ---
with open(DATA_DIR / "benchmark_summary.json") as f:
    summary_full = json.load(f)

with open(DATA_DIR / "benchmark_recursive_summary.json") as f:
    summary_recursive = json.load(f)

with open(DATA_DIR / "benchmark_recursive_images_summary.json") as f:
    summary_images = json.load(f)

# --- Plan DataFrames ---
df_full = pd.read_csv(DATA_DIR / "benchmark_industria.csv")
df_recursive = pd.read_csv(DATA_DIR / "benchmark_recursive_industria.csv")
df_images = pd.read_csv(DATA_DIR / "benchmark_recursive_images_industria.csv")

print(f"benchmark-full:             {len(df_full):>3} planes")
print(f"benchmark-recursive:        {len(df_recursive):>3} planes")
print(f"benchmark-recursive-images: {len(df_images):>3} planes")

## 2. Comparativa de Estrategias - Planes por ISP

In [ ]:
# Construir tabla comparativa desde summaries
isps = [d["isp"] for d in summary_full["details"]]

comparison = pd.DataFrame({
    "ISP": [isp.capitalize() if isp != "cnt" else "CNT" for isp in isps],
    "isp_key": isps,
    "Full": [d["plans_extracted"] for d in summary_full["details"]],
    "Recursive": [d["plans_extracted"] for d in summary_recursive["details"]],
    "Recursive+IMG": [d["plans_extracted"] for d in summary_images["details"]],
})

# Paginas e imagenes del recursivo con imagenes
pages_imgs = {d["isp"]: d for d in summary_images["details"]}
comparison["Paginas"] = comparison["isp_key"].map(lambda k: pages_imgs[k].get("pages_crawled", 0))
comparison["Imagenes"] = comparison["isp_key"].map(lambda k: pages_imgs[k].get("images_analyzed", 0))

comparison[["ISP", "Full", "Recursive", "Recursive+IMG", "Paginas", "Imagenes"]]

In [ ]:
# --- Grafico de barras agrupadas: Planes por ISP y estrategia ---
fig, ax = plt.subplots(figsize=(12, 5))

x = np.arange(len(comparison))
w = 0.25

bars1 = ax.bar(x - w, comparison["Full"], w, label="Full", color=COLORS["accent"], edgecolor="none", zorder=3)
bars2 = ax.bar(x, comparison["Recursive"], w, label="Recursive", color=COLORS["blue"], edgecolor="none", zorder=3)
bars3 = ax.bar(x + w, comparison["Recursive+IMG"], w, label="Recursive+IMG (OCR)", color=COLORS["green"], edgecolor="none", zorder=3)

ax.set_xticks(x)
ax.set_xticklabels(comparison["ISP"], fontsize=10)
ax.set_ylabel("Planes extraidos")
ax.set_title("Planes Extraidos por ISP segun Estrategia")
ax.legend(loc="upper right", framealpha=0.8, facecolor="#1a1d27", edgecolor="#2e3348")
ax.grid(axis="y", linewidth=0.5)
ax.set_axisbelow(True)

# Valor sobre cada barra
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, h + 0.3, str(int(h)),
                    ha="center", va="bottom", fontsize=8, color="#e2e4ed")

plt.tight_layout()
plt.show()

## 3. Tiempos de Ejecucion y Eficiencia

In [ ]:
# Tiempos medidos (del benchmark real ejecutado 2026-04-19)
timing = pd.DataFrame({
    "Estrategia": ["benchmark-full", "benchmark-recursive", "benchmark-recursive-images\n(OCR)"],
    "Tiempo (s)": [207, 480, 935],  # 3m27s, ~8m, 15m35s
    "CPU user (s)": [38.4, 55.0, 89.2],
    "Planes": [18, 17, 33],
    "Costo": ["$0", "$0", "$0"],
})

timing["Planes/min"] = (timing["Planes"] / (timing["Tiempo (s)"] / 60)).round(1)
timing["Eficiencia CPU"] = (timing["Planes"] / timing["CPU user (s)"]).round(2)

timing

In [ ]:
# --- Doble grafico: Tiempo vs Planes ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

labels = ["Full", "Recursive", "Recursive\n+IMG (OCR)"]
colors_bars = [COLORS["accent"], COLORS["blue"], COLORS["green"]]

# Tiempo total
bars = ax1.barh(labels, timing["Tiempo (s)"], color=colors_bars, edgecolor="none", height=0.5)
for bar, val in zip(bars, timing["Tiempo (s)"]):
    mins = int(val // 60)
    secs = int(val % 60)
    ax1.text(bar.get_width() + 10, bar.get_y() + bar.get_height() / 2,
             f"{mins}m {secs}s", va="center", fontsize=9, color="#e2e4ed")
ax1.set_xlabel("Tiempo (segundos)")
ax1.set_title("Tiempo de Ejecucion")
ax1.set_xlim(0, 1100)
ax1.grid(axis="x", linewidth=0.5)
ax1.invert_yaxis()

# Planes extraidos
bars = ax2.barh(labels, timing["Planes"], color=colors_bars, edgecolor="none", height=0.5)
for bar, val in zip(bars, timing["Planes"]):
    ax2.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
             str(int(val)), va="center", fontsize=9, color="#e2e4ed", fontweight="bold")
ax2.set_xlabel("Planes extraidos")
ax2.set_title("Total de Planes")
ax2.set_xlim(0, 40)
ax2.grid(axis="x", linewidth=0.5)
ax2.invert_yaxis()

plt.tight_layout()
plt.show()

## 4. Analisis de Planes - Precios y Velocidades (Mejor Dataset: Recursive+IMG)

In [ ]:
# Usar el dataset mas completo (recursive+images con 33 planes)
df = df_images.copy()
df["precio_plan"] = pd.to_numeric(df["precio_plan"], errors="coerce")
df["velocidad_download_mbps"] = pd.to_numeric(df["velocidad_download_mbps"], errors="coerce")

# Filtrar planes validos con precio y velocidad
df_valid = df.dropna(subset=["precio_plan", "velocidad_download_mbps"])
df_valid = df_valid[df_valid["precio_plan"] > 0]
df_valid["precio_por_mbps"] = (df_valid["precio_plan"] / df_valid["velocidad_download_mbps"]).round(4)

print(f"Planes validos con precio + velocidad: {len(df_valid)}")
print(f"ISPs representados: {df_valid['marca'].nunique()}")
print()
df_valid[["marca", "nombre_plan", "velocidad_download_mbps", "precio_plan", "precio_por_mbps"]].sort_values("precio_por_mbps")

In [ ]:
# --- Scatter: Precio vs Velocidad por ISP ---
fig, ax = plt.subplots(figsize=(11, 6))

for marca in df_valid["marca"].unique():
    subset = df_valid[df_valid["marca"] == marca]
    key = marca.lower()
    color = ISP_COLORS.get(key, COLORS["gray"])
    ax.scatter(subset["velocidad_download_mbps"], subset["precio_plan"],
               s=80, alpha=0.85, label=marca, color=color, edgecolors="white", linewidth=0.5, zorder=3)

ax.set_xlabel("Velocidad Download (Mbps)")
ax.set_ylabel("Precio sin IVA (USD)")
ax.set_title("Precio vs Velocidad por ISP")
ax.legend(loc="upper left", framealpha=0.8, facecolor="#1a1d27", edgecolor="#2e3348")
ax.grid(True, linewidth=0.5)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

## 5. Distribucion de Precios por ISP

In [ ]:
# --- Box plot de precios por ISP ---
marcas_con_datos = df_valid.groupby("marca").filter(lambda g: len(g) >= 2)["marca"].unique()
df_box = df_valid[df_valid["marca"].isin(marcas_con_datos)].copy()

if len(df_box) > 0:
    fig, ax = plt.subplots(figsize=(10, 5))
    
    marcas_sorted = df_box.groupby("marca")["precio_plan"].median().sort_values().index.tolist()
    box_data = [df_box[df_box["marca"] == m]["precio_plan"].values for m in marcas_sorted]
    box_colors = [ISP_COLORS.get(m.lower(), COLORS["gray"]) for m in marcas_sorted]
    
    bp = ax.boxplot(box_data, patch_artist=True, labels=marcas_sorted,
                    medianprops=dict(color="white", linewidth=1.5),
                    whiskerprops=dict(color="#8b8fa7"),
                    capprops=dict(color="#8b8fa7"),
                    flierprops=dict(marker="o", markerfacecolor="#8b8fa7", markersize=4))
    
    for patch, color in zip(bp["boxes"], box_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
        patch.set_edgecolor("white")
        patch.set_linewidth(0.5)
    
    ax.set_ylabel("Precio sin IVA (USD)")
    ax.set_title("Distribucion de Precios por ISP")
    ax.grid(axis="y", linewidth=0.5)
    ax.set_axisbelow(True)
    
    plt.tight_layout()
    plt.show()
else:
    print("No hay suficientes datos para box plot")

## 6. Costo por Mbps - Ranking de Valor

In [ ]:
# --- Precio por Mbps promedio por ISP ---
avg_cost = df_valid.groupby("marca")["precio_por_mbps"].mean().sort_values()

fig, ax = plt.subplots(figsize=(10, 4.5))
bar_colors = [ISP_COLORS.get(m.lower(), COLORS["gray"]) for m in avg_cost.index]
bars = ax.barh(avg_cost.index, avg_cost.values, color=bar_colors, edgecolor="none", height=0.55)

for bar, val in zip(bars, avg_cost.values):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height() / 2,
            f"${val:.3f}", va="center", fontsize=9, color="#e2e4ed")

ax.set_xlabel("Precio promedio por Mbps (USD)")
ax.set_title("Ranking: Costo por Mbps (menor = mejor valor)")
ax.grid(axis="x", linewidth=0.5)
ax.set_axisbelow(True)
ax.invert_yaxis()

plt.tight_layout()
plt.show()

## 7. Cobertura de Campos (Field Coverage)

In [ ]:
# --- Heatmap de campos llenos por ISP ---
key_fields = [
    "nombre_plan", "velocidad_download_mbps", "velocidad_upload_mbps",
    "precio_plan", "precio_plan_descuento", "meses_descuento",
    "costo_instalacion", "comparticion", "pys_adicionales",
    "meses_contrato", "tecnologia", "beneficios_publicitados",
]

coverage_data = {}
for marca in df["marca"].unique():
    subset = df[df["marca"] == marca]
    row = {}
    for field in key_fields:
        if field in subset.columns:
            non_null = subset[field].notna() & (subset[field] != "") & (subset[field] != 0)
            row[field] = non_null.sum() / len(subset) * 100
        else:
            row[field] = 0
    coverage_data[marca] = row

df_coverage = pd.DataFrame(coverage_data).T
df_coverage = df_coverage[key_fields]

fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(df_coverage.values, cmap="RdYlGn", aspect="auto", vmin=0, vmax=100)

ax.set_xticks(range(len(key_fields)))
ax.set_xticklabels([f.replace("_", "\n") for f in key_fields], fontsize=7.5, ha="center")
ax.set_yticks(range(len(df_coverage)))
ax.set_yticklabels(df_coverage.index, fontsize=9)

# Valores en celdas
for i in range(len(df_coverage)):
    for j in range(len(key_fields)):
        val = df_coverage.values[i, j]
        color = "#1a1d27" if val > 60 else "#e2e4ed"
        ax.text(j, i, f"{val:.0f}%", ha="center", va="center", fontsize=7.5, color=color)

ax.set_title("Cobertura de Campos por ISP (% planes con dato)")
plt.colorbar(im, ax=ax, label="% cobertura", shrink=0.8)

plt.tight_layout()
plt.show()

## 8. Mapa de Estrategia Optima por ISP

In [ ]:
# --- Determinar la mejor estrategia por ISP ---
best_strategy = []
for _, row in comparison.iterrows():
    strategies = {"Full": row["Full"], "Recursive": row["Recursive"], "Recursive+IMG": row["Recursive+IMG"]}
    best = max(strategies, key=strategies.get)
    best_val = strategies[best]
    best_strategy.append({
        "ISP": row["ISP"],
        "Full": row["Full"],
        "Recursive": row["Recursive"],
        "Recursive+IMG": row["Recursive+IMG"],
        "Mejor estrategia": best if best_val > 0 else "Ninguna",
        "Max planes": best_val,
    })

df_best = pd.DataFrame(best_strategy)

# Visualizar
fig, ax = plt.subplots(figsize=(10, 4))
strategy_colors = {"Full": COLORS["accent"], "Recursive": COLORS["blue"],
                   "Recursive+IMG": COLORS["green"], "Ninguna": COLORS["gray"]}

bars = ax.barh(df_best["ISP"], df_best["Max planes"],
               color=[strategy_colors[s] for s in df_best["Mejor estrategia"]],
               edgecolor="none", height=0.55)

for bar, (_, row) in zip(bars, df_best.iterrows()):
    if row["Max planes"] > 0:
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
                f'{row["Max planes"]} ({row["Mejor estrategia"]})',
                va="center", fontsize=8.5, color="#e2e4ed")
    else:
        ax.text(0.3, bar.get_y() + bar.get_height() / 2,
                "Sin datos", va="center", fontsize=8.5, color=COLORS["red"])

ax.set_xlabel("Planes extraidos")
ax.set_title("Mejor Estrategia por ISP")
ax.grid(axis="x", linewidth=0.5)
ax.set_axisbelow(True)
ax.invert_yaxis()
ax.set_xlim(0, 25)

# Leyenda manual
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=v, label=k) for k, v in strategy_colors.items() if k != "Ninguna"]
ax.legend(handles=legend_elements, loc="lower right", framealpha=0.8,
          facecolor="#1a1d27", edgecolor="#2e3348", fontsize=8)

plt.tight_layout()
plt.show()

## 9. Resumen Estadistico del Mercado

In [ ]:
# --- Estadisticas del mercado ---
stats = df_valid.groupby("marca").agg(
    planes=("nombre_plan", "count"),
    vel_min=("velocidad_download_mbps", "min"),
    vel_max=("velocidad_download_mbps", "max"),
    precio_min=("precio_plan", "min"),
    precio_max=("precio_plan", "max"),
    precio_medio=("precio_plan", "median"),
    costo_mbps_medio=("precio_por_mbps", "mean"),
).round(3)

stats.columns = ["Planes", "Vel Min (Mbps)", "Vel Max (Mbps)",
                  "Precio Min ($)", "Precio Max ($)", "Precio Mediana ($)",
                  "$/Mbps Medio"]

stats.sort_values("$/Mbps Medio")

## 10. Conclusiones

### Comparativa de Estrategias
| Aspecto | Ganador | Detalle |
|---|---|---|
| **Velocidad** | `benchmark-full` | 3.5min vs 15.5min (4.5x mas rapido) |
| **Cantidad de planes** | `recursive-images` | 33 vs 18 (+83%) |
| **Cobertura ISPs** | `recursive-images` | 5/8 vs 4/8 ISPs |
| **Costo** | Empate | Ambos $0 (FREE) |
| **Netlife** | `benchmark-full` | Unico que activa tabs interactivos (10 vs 0) |
| **Claro/Xtrim/Alfanet** | `recursive-images` | Crawl descubre paginas internas con mas planes |

### Hallazgos Clave
1. **OCR de Tesseract no extrajo ningun plan** de 52 imagenes analizadas. Las imagenes son banners .webp/.svg con diseno grafico, no tablas de texto.
2. **El valor del recursivo esta en el crawl**, no en las imagenes. Los 33 planes vienen 100% del HTML de paginas internas.
3. **La estrategia optima seria combinar**: Full (para Netlife con tabs) + Recursive (para el resto).
4. **Para que imagenes aporten**, se necesita `--mode llm` con vision API.

### Siguiente Paso
Ejecutar `benchmark-recursive-images --mode llm` para comparar la extraccion de imagenes con LLM vision vs OCR local.